In [9]:
import random
import os, cv2, shutil
import numpy as np
import pandas as pd

from PIL import Image
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from albumentations import (Compose, HorizontalFlip, RandomBrightnessContrast,
                            Rotate, Affine, GaussianBlur, GaussNoise, RGBShift,
                            OpticalDistortion, Perspective, Equalize,
                            CoarseDropout, GridDistortion, RandomShadow, CLAHE,
                            ElasticTransform, MotionBlur, ISONoise)


New Augmentations Measurement Section

In [21]:
WORKSPACE   = "/home/admins/rebuild_workspace"
CROPPED_DIR = os.path.join(WORKSPACE, "02_frames_cropped")

# reference sequence: 60 frames of one word, RGB
SEQ_DIR = os.path.join(CROPPED_DIR, "set_01", "bat")
seq = np.stack([cv2.imread(os.path.join(SEQ_DIR, f"{i:02d}.png")) for i in range(1, 61)])
sample = seq[0].copy()

print("sequence:", seq.shape)
print("frame:", sample.shape)

sequence: (60, 80, 112, 3)
frame: (80, 112, 3)


In [25]:
def frame_candidates(t):
    a = {
        "CoarseDropout":     Compose([CoarseDropout(max_holes=6, max_height=12, max_width=12,
                                                    min_holes=2, fill_value=0, p=1.0)]),
        "GridDistortion":    Compose([GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)]),
        "RandomShadow":      Compose([RandomShadow(shadow_roi=(0,0,1,1), num_shadows_lower=1,
                                                   num_shadows_upper=2, p=1.0)]),
        "CLAHEVarying":      Compose([CLAHE(clip_limit=(1.0, 5.0), tile_grid_size=(3,3), p=1.0)]),
        "ElasticTransform":  Compose([ElasticTransform(alpha=1, sigma=50, p=1.0)]),
        "MotionBlur":        Compose([MotionBlur(blur_limit=(5, 11), p=1.0)]),
        "ISONoise":          Compose([ISONoise(color_shift=(0.01,0.05),
                                               intensity=(0.1,0.5), p=1.0)]),
    }
    return a.get(t)

FRAME_TYPES = ["CoarseDropout","GridDistortion","RandomShadow","CLAHEVarying",
               "ElasticTransform","MotionBlur","ISONoise"]

rows = []
for t in FRAME_TYPES:
    try:
        aug = frame_candidates(t)
        a = aug(image=sample.copy())['image']
        b = aug(image=sample.copy())['image']
        rows.append(dict(name=t, kind="frame",
                         shape_ok=(a.shape == sample.shape),
                         stochastic=(not np.array_equal(a, b)),
                         mean_change=round(float(np.abs(a.astype(int)-sample.astype(int)).mean()), 2),
                         frames_changed="n/a", error=""))
    except Exception as e:
        rows.append(dict(name=t, kind="frame", shape_ok=False, stochastic=None,
                         mean_change=None, frames_changed="n/a", error=str(e)[:80]))

print(pd.DataFrame(rows).to_string(index=False))

            name  kind  shape_ok  stochastic  mean_change frames_changed error
   CoarseDropout frame      True        True         3.87            n/a      
  GridDistortion frame      True        True         5.38            n/a      
    RandomShadow frame      True        True        32.58            n/a      
    CLAHEVarying frame      True        True        25.88            n/a      
ElasticTransform frame      True        True         4.26            n/a      
      MotionBlur frame      True        True         2.47            n/a      
        ISONoise frame      True        True         1.27            n/a      


In [26]:
def morph_sequence(s, alpha_range=(0.3, 0.7)):
    """Blend each frame with the next, simulating slower speech."""
    out = s.copy().astype(np.float32)
    for i in range(len(s) - 1):
        a = np.random.uniform(*alpha_range)
        out[i] = a * s[i] + (1 - a) * s[i+1]
    return out.astype(np.uint8)

def drop_and_refill(s, k=8):
    """Drop k frames at random, refill by interpolation - simulates faster speech."""
    n = len(s)
    keep = sorted(np.random.choice(n, n - k, replace=False))
    kept = s[keep].astype(np.float32)
    idx = np.linspace(0, len(kept) - 1, n)
    out = np.empty_like(s, dtype=np.float32)
    for i, f in enumerate(idx):
        lo, hi = int(np.floor(f)), min(int(np.ceil(f)), len(kept)-1)
        w = f - lo
        out[i] = (1-w) * kept[lo] + w * kept[hi]
    return out.astype(np.uint8)

def smooth_drift(s, max_shift=4):
    """Translate each frame by a smoothly varying offset across the sequence."""
    n = len(s)
    t = np.linspace(0, np.pi, n)
    dx = (max_shift * np.sin(t) * np.random.uniform(-1, 1)).astype(int)
    dy = (max_shift * np.sin(t) * np.random.uniform(-1, 1)).astype(int)
    out = np.empty_like(s)
    for i in range(n):
        M = np.float32([[1, 0, dx[i]], [0, 1, dy[i]]])
        out[i] = cv2.warpAffine(s[i], M, (s.shape[2], s.shape[1]),
                                borderMode=cv2.BORDER_REPLICATE)
    return out

def rate_variation(s, strength=0.4):
    """Non-linear resampling - speed varies across the sequence."""
    n = len(s)
    u = np.linspace(0, 1, n)
    warped = u + strength * np.sin(2 * np.pi * u) / (2 * np.pi)
    warped = np.clip(warped, 0, 1) * (n - 1)
    out = np.empty_like(s, dtype=np.float32)
    sf = s.astype(np.float32)
    for i, f in enumerate(warped):
        lo, hi = int(np.floor(f)), min(int(np.ceil(f)), n-1)
        w = f - lo
        out[i] = (1-w) * sf[lo] + w * sf[hi]
    return out.astype(np.uint8)


SEQ_TYPES = {"FrameMorph": morph_sequence, "DropRefill": drop_and_refill,
             "SmoothDrift": smooth_drift, "RateVariation": rate_variation}

seq_rows = []
for name, fn in SEQ_TYPES.items():
    try:
        a = fn(seq.copy())
        b = fn(seq.copy())
        per_frame = np.abs(a.astype(int) - seq.astype(int)).mean(axis=(1,2,3))
        seq_rows.append(dict(name=name, kind="sequence",
                             shape_ok=(a.shape == seq.shape),
                             stochastic=(not np.array_equal(a, b)),
                             mean_change=round(float(per_frame.mean()), 2),
                             frames_changed=int((per_frame > 0.5).sum()),
                             error=""))
    except Exception as e:
        seq_rows.append(dict(name=name, kind="sequence", shape_ok=False, stochastic=None,
                             mean_change=None, frames_changed=None, error=str(e)[:80]))

all_rows = pd.DataFrame(rows + seq_rows)
all_rows.to_csv(f"{WORKSPACE}/logs/augmentation_candidate_measurement_log.csv", index=False)
print(all_rows.to_string(index=False))

            name     kind  shape_ok  stochastic  mean_change frames_changed error
   CoarseDropout    frame      True        True         3.87            n/a      
  GridDistortion    frame      True        True         5.38            n/a      
    RandomShadow    frame      True        True        32.58            n/a      
    CLAHEVarying    frame      True        True        25.88            n/a      
ElasticTransform    frame      True        True         4.26            n/a      
      MotionBlur    frame      True        True         2.47            n/a      
        ISONoise    frame      True        True         1.27            n/a      
      FrameMorph sequence      True        True         2.60             59      
      DropRefill sequence      True        True         5.76             57      
     SmoothDrift sequence      True        True         2.62             42      
   RateVariation sequence      True       False         6.94             58      


Different Augmentation Group Generation Section

In [17]:
LOGS_DIR = "/home/admins/rebuild_workspace/logs"
SPLIT_CSV = "/home/admins/lip_codebase_clean/docs/results_rebuilt_data/session_split_assignment.csv"

split_df = pd.read_csv(SPLIT_CSV)
angle_of = dict(zip(split_df.set_num, split_df.angle))

GROUP_A = {"normal", "up", "down"}
GROUP_B_ANG = {"left", "right", "unknown"}

ORIGINAL_16 = ["Rotate","Translate","Scale","GaussianBlur","GaussNoise","RGBShift",
               "OpticalDistortion","Perspective","HorizontalFlip","Brightness","Contrast",
               "Downsample","Sharpen","Equalize","ProcessCroppedImage","TemporalShift"]

DROPPED = ["TemporalShift","OpticalDistortion","GaussianBlur","Downsample"]
KEPT_12 = [t for t in ORIGINAL_16 if t not in DROPPED]

TEMPORAL_16 = KEPT_12 + ["FrameMorph","DropRefill","SmoothDrift","CoarseDropout"]
PHOTOGEO_16 = KEPT_12 + ["RandomShadow","CLAHEVarying","GridDistortion","ElasticTransform"]

# stochastic membership, from measurement
STOCHASTIC = {"Rotate","Translate","Scale","GaussianBlur","GaussNoise","RGBShift",
              "OpticalDistortion","Perspective","FrameMorph","DropRefill","SmoothDrift",
              "CoarseDropout","RandomShadow","CLAHEVarying","GridDistortion","ElasticTransform"}


def pick_extra(sessions, n_A, n_B, rng):
    a = [s for s in sessions if angle_of[s] in GROUP_A]
    b = [s for s in sessions if angle_of[s] in GROUP_B_ANG]
    # clamp to what each group can supply, rebalance the shortfall
    take_a = min(n_A, len(a))
    take_b = min(n_B, len(b))
    short = (n_A - take_a) + (n_B - take_b)
    if short:
        spare_a = len(a) - take_a
        spare_b = len(b) - take_b
        add_a = min(short, spare_a)
        take_a += add_a
        short -= add_a
        take_b += min(short, spare_b)
    return rng.sample(a, take_a) + rng.sample(b, take_b)


def build_slots(sessions, n_needed, rng):
    """Each session used floor(n/len) times, remainder chosen by angle rule."""
    n = len(sessions)
    base = n_needed // n
    extra = n_needed - base * n
    slots = list(sessions) * base
    if extra:
        half = extra // 2
        slots += pick_extra(sessions, half, extra - half, rng)
    assert len(slots) == n_needed
    return slots


def assign(types_pool, slots, n_needed, rng, uses_per_type):
    types = [t for t in types_pool for _ in range(uses_per_type)]
    assert len(types) == n_needed, (len(types), n_needed)
    for _ in range(500):
        t = types[:]; rng.shuffle(t)
        used, pairs, ok = defaultdict(set), [], True
        for sess, typ in zip(slots, t):
            if typ in used[sess]:
                ok = False; break
            used[sess].add(typ); pairs.append((sess, typ))
        if ok:
            return pairs
    raise RuntimeError("assignment failed")


def assign_rounds(types_pool, slots, n_needed, rng, uses_per_type):
    """Constructive assignment for high uses_per_type, where random
    shuffling cannot satisfy the no-duplicate constraint."""
    from collections import Counter
    types = [t for t in types_pool for _ in range(uses_per_type)]
    assert len(types) == n_needed, (len(types), n_needed)

    remaining = dict(Counter(slots))
    used = defaultdict(set)
    pairs = []
    pool = list(types_pool)

    for round_i in range(uses_per_type):
        rng.shuffle(pool)
        for typ in pool:
            avail = sorted([s for s, c in remaining.items()
                            if c > 0 and typ not in used[s]],
                           key=lambda s: -remaining[s])
            if not avail:
                raise RuntimeError(f"could not place {typ} in round {round_i}")
            s = avail[0]
            used[s].add(typ)
            remaining[s] -= 1
            pairs.append((s, typ))

    assert len(pairs) == n_needed, (len(pairs), n_needed)
    rng.shuffle(pairs)
    return pairs


def build_plan(type_set, train_aug, eval_aug, seed, label):
    rng = random.Random(seed)
    rows = []
    for split, n_aug in [("train", train_aug), ("validation", eval_aug), ("test", eval_aug)]:
        sessions = sorted(split_df.loc[split_df.split == split, "set_num"].tolist())
        slots = build_slots(sessions, n_aug, rng)

        if n_aug == 16:                       # full coverage: one of each
            pairs = assign(type_set, slots, 16, rng, 1)
        elif n_aug % 16 == 0:                 # even multiple
            uses = n_aug // 16
            fn = assign if uses <= 6 else assign_rounds
            pairs = fn(type_set, slots, n_aug, rng, uses)
        else:                                 # 11 slots: 8 stochastic + 3 fixed
            stoch = [t for t in type_set if t in STOCHASTIC]
            fixed = [t for t in type_set if t not in STOCHASTIC and t != "TemporalShift"]
            chosen = stoch + rng.sample(fixed, n_aug - len(stoch))
            pairs = assign(chosen, slots, n_aug, rng, 1)

        for i, (sess, typ) in enumerate(pairs, start=1):
            rows.append(dict(split=split, aug_index=i,
                             aug_set_name=f"aug_{split}_{i:02d}",
                             source_set=sess, source_name=f"set_{sess:02d}",
                             source_angle=angle_of[sess], aug_type=typ))
    df = pd.DataFrame(rows)
    df.to_csv(f"{LOGS_DIR}/augmentation_plan_{label}_log.csv", index=False)
    return df

PLANS = {
    "temporal":       (TEMPORAL_16, 48, 11, 201),
    "photogeo":       (PHOTOGEO_16, 48, 11, 202),
    "volume":         (ORIGINAL_16, 96, 11, 203),
    "full_original":  (ORIGINAL_16, 96, 16, 204),
    "full_temporal":  (TEMPORAL_16, 96, 16, 205),
    "full_photogeo":  (PHOTOGEO_16, 96, 16, 206),
    "vol144":         (ORIGINAL_16, 144, 11, 207),
    "vol192":         (ORIGINAL_16, 192, 11, 208),
    "vol288":         (ORIGINAL_16, 288, 11, 209),
}

for label, (types, tr, ev, seed) in PLANS.items():
    df = build_plan(types, tr, ev, seed, label)
    counts = df.split.value_counts().to_dict()
    ntypes = df[df.split=="validation"].aug_type.nunique()
    print(f"{label:16s} {counts}  val types: {ntypes}  "
          f"session+type dupes: {(df.groupby(['source_set','aug_type']).size() > 1).any()}")

temporal         {'train': 48, 'validation': 11, 'test': 11}  val types: 11  session+type dupes: False
photogeo         {'train': 48, 'validation': 11, 'test': 11}  val types: 11  session+type dupes: False
volume           {'train': 96, 'validation': 11, 'test': 11}  val types: 11  session+type dupes: False
full_original    {'train': 96, 'validation': 16, 'test': 16}  val types: 16  session+type dupes: False
full_temporal    {'train': 96, 'validation': 16, 'test': 16}  val types: 16  session+type dupes: False
full_photogeo    {'train': 96, 'validation': 16, 'test': 16}  val types: 16  session+type dupes: False
vol144           {'train': 144, 'validation': 11, 'test': 11}  val types: 11  session+type dupes: False
vol192           {'train': 192, 'validation': 11, 'test': 11}  val types: 11  session+type dupes: False
vol288           {'train': 288, 'validation': 11, 'test': 11}  val types: 11  session+type dupes: False


In [8]:
for label in ["full_original", "full_temporal", "full_photogeo"]:
    df = pd.read_csv(f"{LOGS_DIR}/augmentation_plan_{label}_log.csv")
    print(f"\n{label}")
    for split in ["train","validation","test"]:
        sub = df[df.split==split]
        grp = sub.source_angle.map(lambda a: "A" if a in GROUP_A else "B")
        print(f"  {split:11s} angle A: {(grp=='A').sum():2d}  angle B: {(grp=='B').sum():2d}")


full_original
  train       angle A: 50  angle B: 46
  validation  angle A:  7  angle B:  9
  test        angle A: 10  angle B:  6

full_temporal
  train       angle A: 50  angle B: 46
  validation  angle A:  7  angle B:  9
  test        angle A: 10  angle B:  6

full_photogeo
  train       angle A: 50  angle B: 46
  validation  angle A:  7  angle B:  9
  test        angle A: 10  angle B:  6


In [18]:
WORKSPACE   = "/home/admins/rebuild_workspace"
CROPPED_DIR = os.path.join(WORKSPACE, "02_frames_cropped")
LOGS_DIR    = os.path.join(WORKSPACE, "logs")
WORDS = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']
TARGET_SIZE = (112, 80)

# ---- frame-level: original 12 kept + 4 photogeo ----
def get_frame_aug(t):
    a = {
        "HorizontalFlip":    Compose([HorizontalFlip(p=1.0)]),
        "Brightness":        Compose([RandomBrightnessContrast(brightness_limit=(0.3,0.3), contrast_limit=0.0, p=1.0)]),
        "Contrast":          Compose([RandomBrightnessContrast(brightness_limit=0.0, contrast_limit=(0.3,0.3), p=1.0)]),
        "Rotate":            Compose([Rotate(limit=5, p=1.0)]),
        "Translate":         Compose([Affine(translate_px={"x":(-10,10),"y":(-10,10)}, p=1.0)]),
        "Scale":             Compose([Affine(scale=(0.9,1.1), p=1.0)]),
        "GaussianBlur":      Compose([GaussianBlur(blur_limit=(7,15), p=1.0)]),
        "GaussNoise":        Compose([GaussNoise(var_limit=(100.0,250.0), p=1.0)]),
        "RGBShift":          Compose([RGBShift(r_shift_limit=50, g_shift_limit=50, b_shift_limit=50, p=1.0)]),
        "OpticalDistortion": Compose([OpticalDistortion(distort_limit=0.2, shift_limit=0.2, p=1.0)]),
        "Perspective":       Compose([Perspective(scale=(0.05,0.1), p=1.0)]),
        "Equalize":          Compose([Equalize(p=1.0)]),
        "CoarseDropout":     Compose([CoarseDropout(max_holes=6, max_height=12, max_width=12,
                                                    min_holes=2, fill_value=0, p=1.0)]),
        "GridDistortion":    Compose([GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)]),
        "RandomShadow":      Compose([RandomShadow(shadow_roi=(0,0,1,1), num_shadows_lower=1,
                                                   num_shadows_upper=2, p=1.0)]),
        "CLAHEVarying":      Compose([CLAHE(clip_limit=(1.0,5.0), tile_grid_size=(3,3), p=1.0)]),
        "ElasticTransform":  Compose([ElasticTransform(alpha=1, sigma=50, p=1.0)]),
    }
    return a.get(t)

def downsample_image(img, scale=0.5):
    h, w = img.shape[:2]
    return cv2.resize(cv2.resize(img, (int(w*scale), int(h*scale))), (w, h))

def sharpen_image(img):
    return cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))

def process_cropped_image(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(3,3)).apply(l)
    out = cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2BGR)
    out = cv2.GaussianBlur(out, (7,7), 0)
    out = cv2.bilateralFilter(out, d=5, sigmaColor=75, sigmaSpace=75)
    out = cv2.filter2D(out, -1, np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]]))
    return cv2.GaussianBlur(out, (5,5), 0)

FRAME_CUSTOM = {"Downsample": downsample_image, "Sharpen": sharpen_image,
                "ProcessCroppedImage": process_cropped_image, "TemporalShift": lambda x: x}

# ---- sequence-level: the 3 temporal types ----
def morph_sequence(s, alpha_range=(0.3, 0.7)):
    out = s.copy().astype(np.float32)
    for i in range(len(s) - 1):
        a = np.random.uniform(*alpha_range)
        out[i] = a * s[i] + (1 - a) * s[i+1]
    return out.astype(np.uint8)

def drop_and_refill(s, k=8):
    n = len(s)
    keep = sorted(np.random.choice(n, n - k, replace=False))
    kept = s[keep].astype(np.float32)
    idx = np.linspace(0, len(kept) - 1, n)
    out = np.empty_like(s, dtype=np.float32)
    for i, f in enumerate(idx):
        lo, hi = int(np.floor(f)), min(int(np.ceil(f)), len(kept)-1)
        w = f - lo
        out[i] = (1-w) * kept[lo] + w * kept[hi]
    return out.astype(np.uint8)

def smooth_drift(s, max_shift=4):
    n = len(s)
    t = np.linspace(0, np.pi, n)
    dx = (max_shift * np.sin(t) * np.random.uniform(-1, 1)).astype(int)
    dy = (max_shift * np.sin(t) * np.random.uniform(-1, 1)).astype(int)
    out = np.empty_like(s)
    for i in range(n):
        M = np.float32([[1,0,dx[i]],[0,1,dy[i]]])
        out[i] = cv2.warpAffine(s[i], M, (s.shape[2], s.shape[1]),
                                borderMode=cv2.BORDER_REPLICATE)
    return out

SEQ_TYPES = {"FrameMorph": morph_sequence, "DropRefill": drop_and_refill,
             "SmoothDrift": smooth_drift}

print("frame types:", len([t for t in get_frame_aug.__wrapped__.__code__.co_consts if 0]) if False else "ok")
print("sequence types:", list(SEQ_TYPES))

frame types: ok
sequence types: ['FrameMorph', 'DropRefill', 'SmoothDrift']


In [19]:
def resize_if_needed(img):
    h, w = img.shape[:2]
    tw, th = TARGET_SIZE
    if (h, w) == (th, tw):
        return img
    interp = cv2.INTER_AREA if (h > th or w > tw) else cv2.INTER_LINEAR
    return cv2.resize(img, (tw, th), interpolation=interp)


def apply_frame_type(img, t):
    if t in FRAME_CUSTOM:
        return FRAME_CUSTOM[t](img)
    return get_frame_aug(t)(image=img)['image']


def make_one_set(args):
    row, aug_dir = args
    src_set, aug_name, aug_type = row['source_name'], row['aug_set_name'], row['aug_type']

    for word in WORDS:
        src = os.path.join(CROPPED_DIR, src_set, word)
        dst = os.path.join(aug_dir, aug_name, word)
        os.makedirs(dst, exist_ok=True)
        files = sorted(f for f in os.listdir(src) if f.endswith('.png'))

        if aug_type in SEQ_TYPES:
            seq = np.stack([cv2.imread(os.path.join(src, f)) for f in files])
            out_seq = SEQ_TYPES[aug_type](seq)
            for idx in range(len(out_seq)):
                cv2.imwrite(os.path.join(dst, f"{idx+1:02d}.png"),
                            resize_if_needed(out_seq[idx]))
        else:
            for idx, f in enumerate(files, start=1):
                img = cv2.imread(os.path.join(src, f))
                cv2.imwrite(os.path.join(dst, f"{idx:02d}.png"),
                            resize_if_needed(apply_frame_type(img, aug_type)))

    return dict(aug_set_name=aug_name, source_set=src_set,
                aug_type=aug_type, split=row['split'], frames=len(WORDS)*60)


VARIANTS = {
    "t":  ("temporal",      "03t_frames_augmented_temporal"),
    "p":  ("photogeo",      "03p_frames_augmented_photogeo"),
    "v":  ("volume",        "03v_frames_augmented_volume"),
    "f":  ("full_original", "03f_frames_augmented_full"),
    "ft": ("full_temporal", "03ft_frames_augmented_full_temporal"),
    "fp": ("full_photogeo", "03fp_frames_augmented_full_photogeo"),
    "h":  ("vol144",        "03h_frames_augmented_vol144"),
    "i":  ("vol192",        "03i_frames_augmented_vol192"),
    "j":  ("vol288",        "03j_frames_augmented_vol288"),
}

for letter, (plan_label, folder) in VARIANTS.items():
    plan = pd.read_csv(f"{LOGS_DIR}/augmentation_plan_{plan_label}_log.csv")
    aug_dir = os.path.join(WORKSPACE, folder)
    tasks = [(r, aug_dir) for r in plan.to_dict('records')]

    start, results = datetime.now(), []
    with ProcessPoolExecutor(max_workers=10) as ex:
        for fut in as_completed([ex.submit(make_one_set, t) for t in tasks]):
            results.append(fut.result())

    gen = pd.DataFrame(results)
    gen.to_csv(f"{LOGS_DIR}/augmentation_generated_{plan_label}_log.csv", index=False)
    print(f"{letter:3s} {plan_label:15s} sets {len(gen):3d}  "
          f"frames {gen.frames.sum():,}  ({(datetime.now()-start).seconds}s)")

t   temporal        sets  70  frames 42,000  (38s)
p   photogeo        sets  70  frames 42,000  (35s)
v   volume          sets 118  frames 70,800  (54s)
f   full_original   sets 128  frames 76,800  (54s)
ft  full_temporal   sets 128  frames 76,800  (49s)
fp  full_photogeo   sets 128  frames 76,800  (61s)
h   vol144          sets 166  frames 99,600  (72s)
i   vol192          sets 214  frames 128,400  (92s)
j   vol288          sets 310  frames 186,000  (134s)


In [20]:
ROWS, COLS = 10, 6
SPLIT_FOLDER = {"train":"Train", "validation":"Validation", "test":"Test"}

def build_grid(args):
    src_dir, set_label, word, grid_dir = args
    wp = os.path.join(src_dir, set_label, word)
    frames = sorted(f for f in os.listdir(wp) if f.endswith('.png'))
    if len(frames) != 60:
        return dict(set_label=set_label, word=word, status=f"COUNT_{len(frames)}")
    first = cv2.imread(os.path.join(wp, frames[0]))
    fh, fw, ch = first.shape
    grid = np.zeros((fh*ROWS, fw*COLS, ch), dtype=np.uint8)
    for i, f in enumerate(frames):
        grid[(i//COLS)*fh:(i//COLS+1)*fh, (i%COLS)*fw:(i%COLS+1)*fw] = cv2.imread(os.path.join(wp, f))
    os.makedirs(grid_dir, exist_ok=True)
    cv2.imwrite(os.path.join(grid_dir, f"{set_label}_{word}.png"), grid)
    return dict(set_label=set_label, word=word, status="OK")

def resize_one(args):
    fname, grid_dir, out_dir = args
    with Image.open(os.path.join(grid_dir, fname)) as im:
        im.resize((224,224), Image.LANCZOS).save(os.path.join(out_dir, fname))
    return fname


NAMES = {
 "t":  ("temporal","04t_grids_temporal","05t_grids_resized_temporal","06t_dataset_temporal","06t_test_real_only_temporal"),
 "p":  ("photogeo","04p_grids_photogeo","05p_grids_resized_photogeo","06p_dataset_photogeo","06p_test_real_only_photogeo"),
 "v":  ("volume","04v_grids_volume","05v_grids_resized_volume","06v_dataset_volume","06v_test_real_only_volume"),
 "f":  ("full_original","04f_grids_full","05f_grids_resized_full","06f_dataset_full","06f_test_real_only_full"),
 "ft": ("full_temporal","04ft_grids_full_temporal","05ft_grids_resized_full_temporal","06ft_dataset_full_temporal","06ft_test_real_only_full_temporal"),
 "fp": ("full_photogeo","04fp_grids_full_photogeo","05fp_grids_resized_full_photogeo","06fp_dataset_full_photogeo","06fp_test_real_only_full_photogeo"),
 "h":  ("vol144","04h_grids_vol144","05h_grids_resized_vol144","06h_dataset_vol144","06h_test_real_only_vol144"),
 "i":  ("vol192","04i_grids_vol192","05i_grids_resized_vol192","06i_dataset_vol192","06i_test_real_only_vol192"),
 "j":  ("vol288","04j_grids_vol288","05j_grids_resized_vol288","06j_dataset_vol288","06j_test_real_only_vol288"),
}

for letter, (label, gdir, rdir, ddir, realdir) in NAMES.items():
    aug_dir  = os.path.join(WORKSPACE, VARIANTS[letter][1])
    grid_dir = os.path.join(WORKSPACE, gdir)
    res_dir  = os.path.join(WORKSPACE, rdir)
    data_dir = os.path.join(WORKSPACE, ddir)
    real_dir = os.path.join(WORKSPACE, realdir)
    os.makedirs(res_dir, exist_ok=True)
    start = datetime.now()

    # grids: real sessions + this variant's augmented sets
    tasks  = [(CROPPED_DIR, s, w, grid_dir) for s in sorted(os.listdir(CROPPED_DIR)) for w in WORDS]
    tasks += [(aug_dir, s, w, grid_dir) for s in sorted(os.listdir(aug_dir)) for w in WORDS]
    with ProcessPoolExecutor(max_workers=10) as ex:
        gres = [f.result() for f in as_completed([ex.submit(build_grid, t) for t in tasks])]
    bad = [r for r in gres if r['status'] != "OK"]

    files = sorted(f for f in os.listdir(grid_dir) if f.endswith('.png'))
    with ProcessPoolExecutor(max_workers=10) as ex:
        list(as_completed([ex.submit(resize_one, (f, grid_dir, res_dir)) for f in files]))

    # assemble: real images first (001-042 train, 001-009 eval), then augmented
    plan = pd.read_csv(f"{LOGS_DIR}/augmentation_plan_{VARIANTS[letter][0]}_log.csv")
    split_of_real = dict(zip(split_df.folder, split_df.split))

    buckets = {}
    for s in sorted(os.listdir(CROPPED_DIR)):
        for w in WORDS:
            buckets.setdefault((split_of_real[s], w), []).append(("real", s, f"{s}_{w}.png"))
    for _, r in plan.iterrows():
        for w in WORDS:
            buckets.setdefault((r.split, w), []).append(("aug", r.aug_set_name, f"{r.aug_set_name}_{w}.png"))

    records = []
    for (split, word), items in buckets.items():
        items.sort(key=lambda x: (x[0] != "real", x[1]))
        dst = os.path.join(data_dir, SPLIT_FOLDER[split], word)
        os.makedirs(dst, exist_ok=True)
        for i, (kind, label_, fname) in enumerate(items, start=1):
            shutil.copy2(os.path.join(res_dir, fname), os.path.join(dst, f"{i:03d}.png"))
            records.append(dict(split=split, word=word, final_number=i, final_name=f"{i:03d}.png",
                                source_type=kind, source_set=label_, source_file=fname))

    pd.DataFrame(records).to_csv(f"{LOGS_DIR}/final_image_mapping_{label}_log.csv", index=False)

    for w in WORDS:
        d = os.path.join(real_dir, w); os.makedirs(d, exist_ok=True)
        srcw = os.path.join(data_dir, "Test", w)
        for i in range(1, 10):
            shutil.copy2(os.path.join(srcw, f"{i:03d}.png"), os.path.join(d, f"{i:03d}.png"))

    counts = {sp: len(os.listdir(os.path.join(data_dir, f, WORDS[0]))) for sp, f in SPLIT_FOLDER.items()}
    print(f"{letter:3s} {label:15s} grids {len(files):4d}  bad {len(bad)}  "
          f"per-word train/val/test: {counts['train']}/{counts['validation']}/{counts['test']}  "
          f"({(datetime.now()-start).seconds}s)")

t   temporal        grids 1300  bad 0  per-word train/val/test: 90/20/20  (13s)
p   photogeo        grids 1300  bad 0  per-word train/val/test: 90/20/20  (14s)
v   volume          grids 1780  bad 0  per-word train/val/test: 138/20/20  (19s)
f   full_original   grids 1880  bad 0  per-word train/val/test: 138/25/25  (20s)
ft  full_temporal   grids 1880  bad 0  per-word train/val/test: 138/25/25  (21s)
fp  full_photogeo   grids 1880  bad 0  per-word train/val/test: 138/25/25  (21s)
h   vol144          grids 2260  bad 0  per-word train/val/test: 186/20/20  (24s)
i   vol192          grids 2740  bad 0  per-word train/val/test: 234/20/20  (32s)
j   vol288          grids 3700  bad 0  per-word train/val/test: 330/20/20  (45s)
